In [1]:
!pip install transformers datasets scikit-learn

Importing Libraries

In [2]:
import pandas as pd
import torch

from sklearn.model_selection import train_test_split

from transformers import DistilBertTokenizerFast
from transformers import DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments

from datasets import Dataset

Creating Medical Dataset

In [19]:
data = {
    "symptoms":[
        "fever cough sore throat",
        "headache nausea sensitivity to light",
        "fever body pain fatigue chills",
        "stomach pain diarrhea vomiting",
        "sneezing runny nose congestion",
        "high fever rash joint pain",
        "frequent urination excessive thirst fatigue",
        "chest pain shortness of breath",
        "itching skin rash redness",
        "blurred vision headache dizziness"
    ],

    "disease":[
        "Flu",
        "Migraine",
        "Flu",
        "Food Poisoning",
        "Common Cold",
        "Dengue",
        "Diabetes",
        "Heart Disease",
        "Allergy",
        "Hypertension"
    ]
}

df = pd.DataFrame(data)

Converting Disease Labels to Numbers

In [4]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

df["label"] = encoder.fit_transform(df["disease"])

Train Test Split

In [5]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["symptoms"],
    df["label"],
    test_size=0.2
)

Loading Tokenizer

In [6]:
tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenize Symptoms

In [7]:
train_encodings = tokenizer(
    train_texts.tolist(),
    truncation=True,
    padding=True
)

val_encodings = tokenizer(
    val_texts.tolist(),
    truncation=True,
    padding=True
)

Converting to Dataset

In [8]:
train_dataset = Dataset.from_dict({
    "input_ids": train_encodings["input_ids"],
    "attention_mask": train_encodings["attention_mask"],
    "labels": train_labels.tolist()
})

val_dataset = Dataset.from_dict({
    "input_ids": val_encodings["input_ids"],
    "attention_mask": val_encodings["attention_mask"],
    "labels": val_labels.tolist()
})

Loading Small Language Model

In [9]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(df["label"].unique())
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Training Configuration

In [29]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy="epoch",
    logging_dir="./logs"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training the Model

In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,No log,2.484005
2,No log,2.476759
3,No log,2.493804


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=6, training_loss=2.150770823160807, metrics={'train_runtime': 8.549, 'train_samples_per_second': 2.807, 'train_steps_per_second': 0.702, 'total_flos': 55891660176.0, 'train_loss': 2.150770823160807, 'epoch': 3.0})

In [34]:
import matplotlib.pyplot as plt
import torch

Prediction Function

In [35]:
def predict(symptoms):

    inputs = tokenizer(
        symptoms,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1)

    top3 = torch.topk(probs, 3)

    diseases = []
    confidences = []

    for i in range(3):

        disease = encoder.inverse_transform(
            [top3.indices[0][i].item()]
        )[0]

        confidence = top3.values[0][i].item()

        diseases.append(disease)
        confidences.append(confidence)

    # Create probability chart
    plt.figure()

    plt.bar(diseases, confidences)

    plt.xlabel("Predicted Disease")
    plt.ylabel("Probability")

    plt.title("Disease Prediction Confidence")

    return plt

Testing the Model

In [14]:
predict_disease("fever chills body pain fatigue")

Predicted Disease: Heart Disease


Building a Medical AI Web App

In [30]:
!pip install gradio

In [36]:
import gradio as gr

interface = gr.Interface(
    fn=predict,

    inputs=gr.Textbox(
        lines=2,
        placeholder="Example: fever headache fatigue cough",
        label="Enter Symptoms"
    ),

    outputs=gr.Plot(),

    title="AI Medical Symptom Checker",

    description="""
Enter symptoms to predict possible diseases using a Small Language Model.

⚠️ This system is for educational purposes only and not intended for medical diagnosis.
""",

    examples=[
        ["fever cough fatigue"],
        ["vomiting nausea stomach pain"],
        ["frequent urination thirst fatigue"],
        ["chest pain shortness of breath"]
    ]
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ea5e31409245e68250.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
